In [18]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import yaml
import numpy as np

def get_project_root():
    return Path().resolve().parent

# This file is the master mapping from city to province, the comprehensivre list with codes
# there are 514 unique districts/cities, and 38 unique provinces
df = pd.read_csv(get_project_root() / "data" / "mapping" / "master_2024.csv")

# These are the manually coded assessments of the index indicators (by Hana)
# otherwise known as Hana's dataset
# there are 509 unique cities here (5 missing are the DKI JAKARTA...)
policy_encoding = pd.read_csv(get_project_root() / "data" / "processed" / "policy_encoding.csv")

# maps from official (master) fullname to/from Hanas encoding
with open(get_project_root() / "data" / "mapping"/"city_mapping.yaml", "r") as f:
    city_mapping = yaml.safe_load(f)

city_mapping_reversed = {v: k for k, v in city_mapping.items()}

In [19]:
# create nested dict version of master mapping from master full name for quick reference
# dont need to run unless master changes
def create_tree_yaml(df):
    prov_tree = defaultdict(list)

    for _, row in df.iterrows():
        prov_tree[row["nama_prov"]].append(row["fullname"])

    output_path = get_project_root() / "data" / "mapping"
    output_path.mkdir(parents=True, exist_ok=True)
    file_path = output_path / "prov_tree.yaml"

    with open(file_path, "w", encoding="utf-8") as f:
        yaml.dump(
            prov_tree,
            f,
            allow_unicode=True,
            sort_keys=True,
            default_flow_style=False
        )

### Policy texts

In [20]:
# look at filenames and missing files in directory
# This is the raw data (policy) documents, not yet translated
base = Path.cwd().parent / "data" / "raw" / "localpolicies"
filenames = policy_encoding['filename'].dropna().unique()

txt_exists = np.array([(base / f).exists() for f in filenames])

pdf_exists = np.array([
    (base / Path(f).with_suffix(".pdf").name).exists()
    for f in filenames
])

print(f"Total expected files: {len(filenames)}")
print(f"TXT exists: {txt_exists.sum()}")
print(f"TXT missing but PDF exists: {((~txt_exists) & pdf_exists).sum()}")
print(f"Missing entirely: {((~txt_exists) & (~pdf_exists)).sum()}")

Total expected files: 198
TXT exists: 149
TXT missing but PDF exists: 6
Missing entirely: 43


In [21]:
filenames[(~txt_exists) & (~pdf_exists)]

<ArrowStringArray>
[                     'ACEH_SUBULUSSALAM_KOTA.txt',
                     'DI_YOGYAKARTA_BANTUL_v2.txt',
                'DI_YOGYAKARTA_GUNUNGKIDUL_v2.txt',
                'DKI_JAKARTA_KEPULAUAN_SERIBU.txt',
                            'JAMBI_SAROLANGUN.txt',
                     'JAWA_BARAT_BANDUNG_KOTA.txt',
                           'JAWA_BARAT_BEKASI.txt',
                   'JAWA_BARAT_BEKASI_KOTA_v2.txt',
                     'JAWA_BARAT_CIREBON_KOTA.txt',
                         'JAWA_BARAT_SUMEDANG.txt',
                  'JAWA_TENGAH_KARANGANYAR_v2.txt',
                         'JAWA_TENGAH_KEBUMEN.txt',
                       'JAWA_TENGAH_PURWOREJO.txt',
                'JAWA_TENGAH_SEMARANG_KOTA_v2.txt',
                  'JAWA_TENGAH_SURAKARTA_KOTA.txt',
              'KALIMANTAN_SELATAN_TANAH_BUMBU.txt',
                  'KALIMANTAN_TENGAH_KATINGAN.txt',
              'KALIMANTAN_TENGAH_PULANG_PISAU.txt',
                  'KALIMANTAN_TENGAH_SUKAMARA

In [22]:
policy_encoding[policy_encoding['policy_change']==1]

,fullname,year,policy_version,1.1,1.2,1.3,1.4,1.5,1.6,1.7,...,7.4,7.5,adoption_isu,vaw_reported_cases,gdp,policy_change,has_policy,filename,value_kab,ADM2_PCODE
304,ACEH BIREUEN,2022,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,25.0,15470.0,1,1,ACEH_BIREUEN.txt,1110,ID1110
503,ACEH SUBULUSSALAM KOTA,2023,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,1.0,NaN,7.0,2560.0,1,1,ACEH_SUBULUSSALAM_KOTA.txt,1175,ID1175
515,BALI BADUNG,2013,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,NaN,31790.0,1,1,BALI_BADUNG.txt,5103,ID5103
540,BALI BANGLI,2016,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,NaN,5457.0,1,1,BALI_BANGLI.txt,5106,ID5106
565,BALI BULELENG,2019,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,18.0,35362.0,1,1,BALI_BULELENG.txt,5108,ID5108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10467,SUMATERA SELATAN PRABUMULIH KOTA,2021,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,30.0,8346.0,1,1,SUMATERA_SELATAN_PRABUMULIH_KOTA.txt,1672,ID1672
10506,SUMATERA UTARA BATU BARA,2016,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,NaN,27555.0,1,1,SUMATERA_UTARA_BATU_BARA.txt,1219,ID1219
10689,SUMATERA UTARA LABUHANBATU SELATAN,2023,1,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,1.0,0.0,NaN,6.0,37584.0,1,1,SUMATERA_UTARA_LABUHANBATU_SELATAN.txt,1222,ID1222
10929,SUMATERA UTARA PADANGSIDIMPUAN KOTA,2021,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,NaN,19.0,6784.0,1,1,SUMATERA_UTARA_PADANGSIDIMPUAN_KOTA.txt,1277,ID1277


In [23]:
# specifically, which texts are missing?
missing_mask = (~txt_exists) & (~pdf_exists)
print("\nMissing files:")
for f in filenames[missing_mask]:
    print(f)


Missing files:
ACEH_SUBULUSSALAM_KOTA.txt
DI_YOGYAKARTA_BANTUL_v2.txt
DI_YOGYAKARTA_GUNUNGKIDUL_v2.txt
DKI_JAKARTA_KEPULAUAN_SERIBU.txt
JAMBI_SAROLANGUN.txt
JAWA_BARAT_BANDUNG_KOTA.txt
JAWA_BARAT_BEKASI.txt
JAWA_BARAT_BEKASI_KOTA_v2.txt
JAWA_BARAT_CIREBON_KOTA.txt
JAWA_BARAT_SUMEDANG.txt
JAWA_TENGAH_KARANGANYAR_v2.txt
JAWA_TENGAH_KEBUMEN.txt
JAWA_TENGAH_PURWOREJO.txt
JAWA_TENGAH_SEMARANG_KOTA_v2.txt
JAWA_TENGAH_SURAKARTA_KOTA.txt
KALIMANTAN_SELATAN_TANAH_BUMBU.txt
KALIMANTAN_TENGAH_KATINGAN.txt
KALIMANTAN_TENGAH_PULANG_PISAU.txt
KALIMANTAN_TENGAH_SUKAMARA.txt
KALIMANTAN_TIMUR_KUTAI_TIMUR.txt
KALIMANTAN_TIMUR_PENAJAM_PASER_UTARA.txt
KEPULAUAN_BANGKA_BELITUNG_BANGKA_BARAT_v2.txt
KEPULAUAN_BANGKA_BELITUNG_BANGKA_SELATAN_v2.txt
LAMPUNG_PESISIR_BARAT.txt
LAMPUNG_TANGGAMUS.txt
NUSA_TENGGARA_BARAT_DOMPU.txt
NUSA_TENGGARA_BARAT_LOMBOK_TENGAH_v2.txt
NUSA_TENGGARA_TIMUR_TIMOR_TENGAH_SELATAN.txt
RIAU_BENGKALIS.txt
RIAU_PEKANBARU_KOTA.txt
SULAWESI_BARAT_MAMUJU.txt
SULAWESI_SELATAN_BULUKUMBA.txt
S

In [24]:
# compare two raw sources xlsx

In [25]:
file_path = (
    Path().resolve().parent.parent
    / "Quality of VAW Policy Dataset (Local Policy in Indonesia simple scoring).xlsx"
)
df_1 = pd.read_excel(file_path, sheet_name="Dataset (all)")

file_path = (
    Path().resolve().parent.parent
    / "2004-2025 Updated Quality of VAW Policy Dataset Indonesia.xlsx"
)
df_2 = pd.read_excel(file_path, sheet_name="Dataset")

In [26]:
df_1 =df_1[df_1['Does the city/regency have local regulation on women\'s protection?']=='Yes']

In [27]:
# Capitalise City/Regency values in df_2
df_2["City/Regency"] = df_2["City/Regency"].str.upper()

# Keep only rows with a non-missing OVERALL SCORE
df_2 = df_2.rename(columns={"TOTAL POLICY SCORE": "OVERALL SCORE"})
df_2 = df_2[df_2["OVERALL SCORE"].notna()].copy()

### FILTERED ONLY FOR ORIGINAL POLICY DOCUMENTS
#df_2  =df_2[df_2['POLICY ADOPTION']==1]
#df_2  =df_2[df_2['POLICY ADOPTION']==2]

In [28]:
df_2.head()

,City/Regency,Province,Year,POLICY ADOPTION,OVERALL SCORE,SCOPE OF VIOLENCE,1. Domestic violence,2. Sexual violence,3. Physical violence,4. Psychological violence,...,BUDGET/ FUNDING SOURCES,1. Local government budget,2. National government budget,3. International support/ donors,4. Other legal sources,5. Provincial government budget,Adoption of regulation establishing an Integrated Service Unit for Women and Children's Protection,Number of reported VAW cases (SIMFONI),Gross Regional Domestic Product (GDRP) at current market prices (billion Rupiah),New city/regency year of legal establishment
216,BIREUEN,Aceh,2022,1,38.0,7.0,1.0,1.0,1.0,1.0,...,2.0,1.0,0.0,0.0,1.0,0.0,NaN,25.0,15470,NaN
503,KOTA SUBULUSSALAM,Aceh,2023,1,36.0,6.0,1.0,1.0,1.0,1.0,...,3.0,1.0,1.0,0.0,1.0,1.0,NaN,7.0,2560,NaN
849,PAKPAK BHARAT,North Sumatera,2017,1,30.0,6.0,1.0,1.0,1.0,1.0,...,2.0,1.0,0.0,0.0,1.0,0.0,NaN,0.0,1000,NaN
914,BATU BARA,North Sumatera,2016,1,24.0,5.0,1.0,1.0,1.0,1.0,...,2.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN,27555,NaN
987,LABUHAN BATU SELATAN,North Sumatera,2023,1,39.0,8.0,1.0,1.0,1.0,1.0,...,2.0,1.0,0.0,0.0,1.0,0.0,NaN,6.0,37584,NaN


In [29]:
map_df1_df2={
'DHARMAS RAYA':'DHARMASRAYA',
'KEPULAUAN TANIMBAR (MALUKU TENGGARA BARAT)':"Maluku Tenggara Barat",
'KOTA PADANGSIDEMPUAN':'KOTA PADANG SIDEMPUAN',
"PASANGKAYU (MAMUJU UTARA)":"MAMUJU UTARA",
"PENUKAL ARAB LEMATANG ILIR": "Penukal Abab Lematang Ilir",
}

In [30]:
# Standardise names in both datasets
df_1["City/Regency Name"] = (
    df_1["City/Regency Name"]
    .astype(str)
    .str.strip()
    .str.upper()
)
df_1["City/Regency Name"] = df_1["City/Regency Name"].replace(map_df1_df2)
df_1["City/Regency Name"] = (
    df_1["City/Regency Name"]
    .astype(str)
    .str.strip()
    .str.upper()
)
df_2["City/Regency"] = (
    df_2["City/Regency"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Merge scores
comparison = (
    df_1[["City/Regency Name", "OVERALL SCORE"]]
    .rename(columns={"OVERALL SCORE": "score_df1"})
    .merge(
        df_2[["City/Regency", "Year", "OVERALL SCORE"]]
        .rename(columns={"OVERALL SCORE": "score_df2"}),
        left_on="City/Regency Name",
        right_on="City/Regency",
        how="outer",
        indicator=True
    )
)

# Compare scores
comparison["match"] = comparison["score_df1"] == comparison["score_df2"]

# Summary
matched = (
    (comparison["_merge"] == "both")
    & comparison["match"]
).sum()

mismatched = (
    (comparison["_merge"] == "both")
    & ~comparison["match"]
).sum()

only_df1 = (comparison["_merge"] == "left_only").sum()
only_df2 = (comparison["_merge"] == "right_only").sum()

print("SUMMARY")
print(f"Matched scores: {matched}")
print(f"Mismatched scores: {mismatched}")
print(f"Only in df_1: {only_df1}")
print(f"Only in df_2: {only_df2}")

# Print mismatches
mismatch_rows = comparison[
    (comparison["_merge"] == "both")
    & ~comparison["match"]
][["City/Regency Name", "score_df1", "score_df2"]]

print("\nMISMATCHED SCORES")
print(mismatch_rows.to_string(index=False))

# Print matches
# match_rows = comparison[
#     (comparison["_merge"] == "both")
#     & comparison["match"]
# ][["City/Regency Name", "Year", "score_df1", "score_df2"]]

# print("\nMATCHED SCORES")
# print(match_rows.to_string(index=False))

# Optional: print names missing from either file
print("\nONLY IN df_1 (infirst version, missing from second version)")
print(
    comparison.loc[
        comparison["_merge"] == "left_only",
        "City/Regency Name"
    ].to_string(index=False)
)

print("\nONLY IN df_2 (since added from first version)")
print(
    comparison.loc[
        comparison["_merge"] == "right_only",
        ["City/Regency", "Year"]
    ].to_string(index=False)
)
# only in df_2 should have dates >=2022

SUMMARY
Matched scores: 159
Mismatched scores: 9
Only in df_1: 1
Only in df_2: 30

MISMATCHED SCORES
City/Regency Name  score_df1  score_df2
   BANGKA SELATAN       27.0       28.0
           BANTUL       30.0       33.0
     GUNUNG KIDUL       32.0       33.0
      KARANGANYAR       27.0       35.0
      KOTA BEKASI       37.0       39.0
    KOTA SEMARANG       35.0       36.0
    LOMBOK TENGAH       30.0       32.0
       LUWU TIMUR       30.0       28.0
OGAN KOMERING ULU       32.0       27.0

ONLY IN df_1 (infirst version, missing from second version)
TIMOR TENGAH UTARA

ONLY IN df_2 (since added from first version)
        City/Regency   Year
           BENGKALIS 2022.0
           BULUKUMBA 2024.0
         BUTON UTARA 2022.0
               DOMPU 2024.0
            KATINGAN 2023.0
             KEBUMEN 2025.0
  KEPULAUAN MENTAWAI 2025.0
        KONAWE UTARA 2022.0
        KOTA BANDUNG 2025.0
        KOTA CIREBON 2024.0
      KOTA PALEMBANG 2022.0
      KOTA PEKANBARU 2023.0
   KOTA 

In [31]:
df_1[df_1["City/Regency Name"]=='TIMOR TENGAH SELATAN']

,No,City/Regency Name,Jurisdiction Type (City/Regency),Province,Does the city/regency have local regulation on women's protection?,When was the women's protection regulation adopted?,OVERALL SCORE,SCOPE OF VIOLENCE,1. Domestic violence,2. Sexual violence,...,10. Refugee / forced displacement,11. Domestic worker,12. Women experiencing houselessness,13. Formerly incarcerated women,BUDGET/ FUNDING SOURCES,1. Local government budget,2. National government budget,3. International support/ donors,4. Other legal sources,5. Provincial government budget


In [32]:
df_2[df_2["City/Regency"]=="Timor Tengah Selatan"]

,City/Regency,Province,Year,POLICY ADOPTION,OVERALL SCORE,SCOPE OF VIOLENCE,1. Domestic violence,2. Sexual violence,3. Physical violence,4. Psychological violence,...,BUDGET/ FUNDING SOURCES,1. Local government budget,2. National government budget,3. International support/ donors,4. Other legal sources,5. Provincial government budget,Adoption of regulation establishing an Integrated Service Unit for Women and Children's Protection,Number of reported VAW cases (SIMFONI),Gross Regional Domestic Product (GDRP) at current market prices (billion Rupiah),New city/regency year of legal establishment


In [33]:
list(df_2["City/Regency"])

['BIREUEN',
 'KOTA SUBULUSSALAM',
 'PAKPAK BHARAT',
 'BATU BARA',
 'LABUHAN BATU SELATAN',
 'KOTA PADANG SIDEMPUAN',
 'KEPULAUAN MENTAWAI',
 'PESISIR SELATAN',
 'SOLOK',
 'PADANG PARIAMAN',
 'DHARMASRAYA',
 'KOTA BUKITTINGGI',
 'KOTA PAYAKUMBUH',
 'BENGKALIS',
 'KOTA PEKANBARU',
 'SAROLANGUN',
 'BUNGO',
 'OGAN KOMERING ULU',
 'OGAN KOMERING ULU',
 'MUARA ENIM',
 'MUSI RAWAS',
 'MUSI BANYUASIN',
 'OGAN KOMERING ULU SELATAN',
 'PENUKAL ABAB LEMATANG ILIR',
 'MUSI RAWAS UTARA',
 'KOTA PALEMBANG',
 'KOTA PRABUMULIH',
 'REJANG LEBONG',
 'SELUMA',
 'KEPAHIANG',
 'TANGGAMUS',
 'LAMPUNG TIMUR',
 'LAMPUNG UTARA',
 'PESAWARAN',
 'PRINGSEWU',
 'PESISIR BARAT',
 'KOTA BANDAR LAMPUNG',
 'KOTA METRO',
 'BANGKA',
 'BELITUNG',
 'BANGKA BARAT',
 'BANGKA BARAT',
 'BANGKA TENGAH',
 'BANGKA SELATAN',
 'BANGKA SELATAN',
 'KOTA PANGKAL PINANG',
 'KARIMUN',
 'DKI JAKARTA',
 'BANDUNG',
 'GARUT',
 'TASIKMALAYA',
 'CIREBON',
 'MAJALENGKA',
 'SUMEDANG',
 'INDRAMAYU',
 'PURWAKARTA',
 'KARAWANG',
 'BEKASI',
 'BAND